<a href="https://colab.research.google.com/github/wszd158/AIchat/blob/main/llama3.1-8B-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# We have to check which Torch version for Xformers (2.3 -> 0.0.27)
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/235 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
from datasets import load_from_disk

print("加载数据集...")
dataset_path = "/content/Novel_chatml"
dataset = load_from_disk(dataset_path)
print(f"原始数据量：{len(dataset)} 条")

# ========== 数据预处理 ==========
def is_valid_messages(example):
    messages = example.get("messages")
    return messages and isinstance(messages, list) and len(messages) > 0

dataset = dataset.filter(is_valid_messages)
print(f"过滤后有效数据量：{len(dataset)} 条")

# ChatML 格式化函数（不依赖 tokenizer.apply_chat_template）
EOS_TOKEN = tokenizer.eos_token  # 一般是 "</s>"

def format_and_truncate(example):
    messages = example["messages"]

    # 按角色拼成 ChatML 格式
    parts = []
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        # 常见的 ChatML 格式：
        # <|im_start|>system
        # {system text}<|im_end|>
        # <|im_start|>user
        # {user text}<|im_end|>
        # <|im_start|>assistant
        # {assistant text}<|im_end|>
        parts.append(f"<|im_start|>{role}\n{content}<|im_end|>")
    text = "\n".join(parts) + EOS_TOKEN

    # 如果太长，截断
    tokens = tokenizer(text, truncation=False, add_special_tokens=True)["input_ids"]
    if len(tokens) > max_seq_length:
        truncated_tokens = tokens[:max_seq_length-1] + [tokenizer.eos_token_id]
        text = tokenizer.decode(truncated_tokens, skip_special_tokens=False)

    return {"text": text}

dataset = dataset.map(format_and_truncate, remove_columns=dataset.column_names)
print(f"格式化后数据量：{len(dataset)} 条")

# 抽样检查
for i in range(min(2, len(dataset))):
    sample = dataset[i]["text"]
    tokens = tokenizer(sample, add_special_tokens=True)["input_ids"]
    print(f"样本 {i+1} (长度: {len(tokens)}): {sample[:300]}...")

加载数据集...
原始数据量：1344 条
过滤后有效数据量：1322 条


Map:   0%|          | 0/1322 [00:00<?, ? examples/s]

格式化后数据量：1322 条
样本 1 (长度: 2049): <|begin_of_text|><|im_start|>assistant
索娜说到小不点，敢坏我们赖利哥的好事，你怕是活腻了，老娘我今天就好好教训教训你一下，说着冲了过来。<|im_end|>
<|im_start|>user
小霞不慌不忙地冲上前，对着索娜的大腿就是一切，一瞬间索娜的整条大腿飞了出去，索娜疼苦地大叫一声。<|im_end|>
<|im_start|>assistant
小霞接着拿出电钻，对着索娜那个已经被赖利哥抽烂的阴部就钻了进去，又从索娜的口中钻出来，再在空中使出了三连斩，直接把索娜切成了几块巨大的肉块。<|im_end|>
<|im_start|>assistant
...
样本 2 (长度: 1919): <|im_start|>assistant
小狐狸缓缓走到溪边，急促流动的溪水明净而清亮，发出好听的水流声，为幽静的丛林增添着一笔生机。<|im_end|>
<|im_start|>user
我低头畅饮着溪水，宝石般明亮的双眼映在水面里，仿佛自己也成了水中的一道风景。<|im_end|>
<|im_start|>assistant
她没有顾及水中自己动人的倒影，只管惬意地饮水，随后在如棉席柔软的草地上挑了个舒服的位置，蜷缩着伏下身，懒洋洋合上了那双让人喜爱的双眼。<|im_end|>
<|im_start|>user
一身绒黄，毛色蓬松柔软，尤其在身后那根毛绒大尾巴上更显得生动可爱。<|im_...


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1322 [00:00<?, ? examples/s]

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
5.787 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,322 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,2.131624
2,2.233456
3,2.223503
4,2.093152
5,1.950182
6,1.940181
7,2.047083
8,1.951734
9,2.168526
10,1.920610


In [ ]:
model.save_pretrained("lora_model") # Local saving
tokenizer.save_pretrained("lora_model")
# model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json', 'lora_model/tokenizer.json')

In [ ]:
# ============================================
# Colab GGUF 量化导出脚本 (Q4_K_M)
# ============================================
# 1. 安装依赖（如果尚未安装）
!pip install -q unsloth
!pip install -q "huggingface_hub[cli]"  # 用于后续下载
# 2. 导入库
from unsloth import FastLanguageModel
import torch
# ============================================
# 配置参数
# ============================================
# LoRA 模型路径（你之前训练保存的路径）
lora_model_path = "/content/lora_model"  # 如果在当前目录，否则填完整路径如 "/content/outputs/lora_model"
# 基础模型路径（训练时用的基座模型，用于合并 LoRA）
base_model_path = "unsloth/Meta-Llama-3.1-8B"  # 替换为你实际用的基座模型
# 导出配置
output_dir = "gguf_export"          # 导出目录
quantization_method = "q4_k_m"      # Q4_K_M 量化
model_name = "my-model-q4km"        # 导出的文件名前缀
# ============================================
# 3. 加载 LoRA 模型（合并到基座）
# ============================================
print("正在加载 LoRA 模型并合并到基座...")
# 先加载基础模型
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=False,  # 合并时不要用 4bit，需要完整精度
)
# 加载 LoRA 权重并合并
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=3407,
)
model.load_adapter(lora_model_path, adapter_name="default")
model.set_adapter("default")
# 合并 LoRA 到基座（关键步骤，减少显存占用）
print("正在合并 LoRA 权重...")
model = model.merge_and_unload()  # 合并后不再需要 PEFT
print("模型合并完成！")
# ============================================
# 4. 导出为 GGUF (Q4_K_M)
# ============================================
print(f"\n正在导出为 GGUF 格式 (量化方法: {quantization_method})...")
# 创建输出目录
import os
os.makedirs(output_dir, exist_ok=True)
# 使用 Unsloth 的 save_pretrained_gguf 方法
model.save_pretrained_gguf(
    output_dir,           # 导出目录
    tokenizer,            # tokenizer
    quantization_method=quantization_method,  # "q4_k_m"
    # 可选：指定文件名，默认是 unsloth.Q4_K_M.gguf
    # filename="my-custom-name.gguf"
)
print(f"\n✅ 导出完成！文件保存在: {output_dir}/")
# ============================================
# 5. 列出导出文件并提供下载链接
# ============================================
print("\n导出的文件列表：")
!ls -lh {output_dir}/
# 获取具体的 gguf 文件名
gguf_files = [f for f in os.listdir(output_dir) if f.endswith('.gguf')]
if gguf_files:
    gguf_filename = gguf_files[0]
    print(f"\n🎯 GGUF 文件: {gguf_filename}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 127.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 115.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

ModuleNotFoundError: No module named 'unsloth'